# Phase 3: Generalization & Ablation

**Goal:** Evaluate Phase 2 v2 fine-tuned model on held-out generalization set and perform ablation attribution.

**Note:** No retraining occurs in this notebook. It consumes existing Phase 2 v2 artifacts.

In [ ]:
# 2. Runtime/hardware check
import torch
import os
import sys

if not torch.cuda.is_available():
    print("CRITICAL: CUDA unavailable. Colab T4 runtime required.")
    # sys.exit(1) # Stop in notebook context

gpu_name = torch.cuda.get_device_name(0)
if "T4" not in gpu_name:
    print(f"WARNING: Running on {gpu_name}. Project optimized for Tesla T4.")

In [ ]:
# 3. Drive/repo artifact setup
from google.colab import drive
import os

DRIVE_ROOT = "/content/drive/MyDrive/codepause_phase2_v2"
LOCAL_ROOT = "drive/codepause_phase2_v2"

root_path = DRIVE_ROOT if os.path.exists(DRIVE_ROOT) else LOCAL_ROOT
print(f"Using root path: {root_path}")

required_files = [
    "results/phase2_main_report.md",
    "results/phase2_main_baseline.jsonl",
    "results/phase2_main_finetuned.jsonl",
    "results/metadata_run.json",
    "phase2_recipe_a/adapter_config.json",
    "phase2_recipe_a/adapter_model.safetensors"
]

missing = []
for f in required_files:
    p = os.path.join(root_path, f)
    if not os.path.exists(p):
        missing.append(p)

if missing:
    print("CRITICAL: Missing required Phase 2 v2 artifacts:")
    for m in missing:
        print(f"  - {m}")
    # raise FileNotFoundError("Missing required artifacts")
else:
    print("All required artifacts found.")

In [ ]:
# 4. Phase 2 v2 summary
import json

metadata_path = os.path.join(root_path, "results/metadata_run.json")
with open(metadata_path, "r") as f:
    metadata = json.load(f)

print("--- Phase 2 v2 Summary ---")
print(f"Base Model: Qwen/Qwen1.5-1.8B-Chat")
print(f"Adapter: {os.path.join(root_path, 'phase2_recipe_a')}")
print(f"Measured Result: 24/30 -> 25/30 (+3.3pp)")
print(f"Metadata Device: {metadata.get('device', 'unknown')}")
print(f"Caveat: Recorded +3.3pp differs from some earlier summaries claiming +10pp.")

In [ ]:
# 5. Dataset validation
HELD_OUT_PATH = "data/heldout_problems_30.jsonl"

# Check if file exists, if not, try to locate it or create placeholder
if not os.path.exists(HELD_OUT_PATH):
    print(f"ERROR: {HELD_OUT_PATH} missing. Please ensure it is present in 'data/'.")
else:
    !python eval/validate_dataset.py {HELD_OUT_PATH} --schema problems

In [ ]:
# 6. Adapter probe
adapter_path = os.path.join(root_path, "phase2_recipe_a")
!python eval/adapter_probe.py --adapter_path {adapter_path}

## 7. Held-out Evaluation

In [ ]:
# Baseline vs Fine-tuned on Held-out
!python eval/evaluator.py --dataset {HELD_OUT_PATH} --output results/phase3_heldout_baseline.jsonl --model Qwen/Qwen1.5-1.8B-Chat
!python eval/evaluator.py --dataset {HELD_OUT_PATH} --output results/phase3_heldout_finetuned.jsonl --model Qwen/Qwen1.5-1.8B-Chat --adapter {adapter_path}

!python eval/make_report.py --baseline results/phase3_heldout_baseline.jsonl --finetuned results/phase3_heldout_finetuned.jsonl --output results/phase3_heldout_report.md

## 10. Ablation Studies

- **Arm A:** Base Model + Baseline Prompt
- **Arm B:** Base Model + ThinkAnywhere Prompt
- **Arm C:** Adapter + ThinkAnywhere Prompt
- **Arm D:** Adapter + `minimal_python_function` Prompt

In [ ]:
# Run Ablation Arms
!python eval/evaluator.py --dataset {HELD_OUT_PATH} --output results/phase3_ablation_a.jsonl --model Qwen/Qwen1.5-1.8B-Chat --template baseline
!python eval/evaluator.py --dataset {HELD_OUT_PATH} --output results/phase3_ablation_b.jsonl --model Qwen/Qwen1.5-1.8B-Chat --template thinkanywhere
!python eval/evaluator.py --dataset {HELD_OUT_PATH} --output results/phase3_ablation_c.jsonl --model Qwen/Qwen1.5-1.8B-Chat --adapter {adapter_path} --template thinkanywhere
!python eval/evaluator.py --dataset {HELD_OUT_PATH} --output results/phase3_ablation_d.jsonl --model Qwen/Qwen1.5-1.8B-Chat --adapter {adapter_path} --template minimal_python_function

# Ablation Summary would go here
print("Ablation runs completed.")

In [ ]:
# 12. Failure Taxonomy (Phase 2 Remaining)
phase2_finetuned = os.path.join(root_path, "results/phase2_main_finetuned.jsonl")
print(f"Analyzing failures in {phase2_finetuned}...")
# !python eval/analyze_failures.py --results {phase2_finetuned} --output results/phase3_failure_taxonomy_phase2_remaining.md

In [ ]:
# 14. Final Checklist
import os

outputs = [
    "results/phase3_heldout_report.md",
    "results/phase3_heldout_baseline.jsonl",
    "results/phase3_heldout_finetuned.jsonl",
    "results/phase3_ablation_a.jsonl",
    "results/phase3_ablation_b.jsonl",
    "results/phase3_ablation_c.jsonl",
    "results/phase3_ablation_d.jsonl"
]

print("--- Final Checklist ---")
for out in outputs:
    status = "OK" if os.path.exists(out) else "MISSING"
    print(f"{out}: {status}")